In [ ]:
!pip install transformers datasets evaluate accelerate underthesea dotenv huggingface_hub py_vncorenlp -q

In [ ]:
import os
import evaluate
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import py_vncorenlp
import torch.nn as nn
from dotenv import find_dotenv, load_dotenv
from datasets import load_dataset, DatasetDict
from huggingface_hub import login, upload_folder
from scipy.special import softmax
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    matthews_corrcoef,
    brier_score_loss
)
from sklearn.calibration import calibration_curve
from torch.utils.data import DataLoader
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
root_path = "/content/drive/MyDrive/BTL_AI_13_IT2302/AIWritingIndicator-13/SourceCode/"
%cd {root_path}
!pwd

/content/drive/.shortcut-targets-by-id/17JBaRqvfDFY0j8cN0vwu8kzqwtDISdDR/BTL_AI_13_IT2302/AIWritingIndicator-13/SourceCode
/content/drive/.shortcut-targets-by-id/17JBaRqvfDFY0j8cN0vwu8kzqwtDISdDR/BTL_AI_13_IT2302/AIWritingIndicator-13/SourceCode


In [ ]:
env_path = find_dotenv()

if env_path:
    load_dotenv(env_path)
else:
    print("Không tìm thấy file .env!")

In [ ]:
hf_token = os.getenv("HF_TOKEN")

if hf_token:
    login()
else:
    print("Không tồn tại env HF_TOKEN!")

In [ ]:
class MyConfig:
  model_name = "vinai/phobert-large"
  seed = 123
  save_dir = "DL_Artifacts/phobert-ai-news-detector"

  class Hub:
    push_to_hub = True
    repo_id = "JuniorThanh/phobert-large-vietnamese-news-ai-detection"
    token = None

  class Tokenizer:
    padding = "max_length"
    max_length = 256
    stride = 64
    truncation = False
    add_special_tokens = False
    padding = False
    return_attention_mask = True

  class Model:
    freeze_layers = 10
    d_features = 768
    d_model = 256
    intermediate_dims = 512
    dropout = 0.2
    num_labels = 2

  class Train:
    lr = 2e-6
    epochs = 5
    batch_size = 16
    weight_decay = 0.05
    warmup_ratio = 0.15
    metric_for_best_model = "eval_id_val_mcc"
    label_column = "label"
    class_weights = [1.0, 1.1]
    freeze_layers = 10
    lr_scheduler_type = "cosine"

  class Inference:
    batch_size = 16

config = MyConfig()

In [ ]:
dataset = load_dataset("kngann2201/vietnamese-ai-detection")

for split in dataset.keys():
    labels = dataset[split]['label']
    count_0 = labels.count(0)
    count_1 = labels.count(1)

    print(f"Tập {split}:")
    print(f"  - Label 0 (Người): {count_0}")
    print(f"  - Label 1 (AI):    {count_1}")
    print(f"  - Tổng cộng:       {len(labels)}")

Tập train:
  - Label 0 (Người): 6059
  - Label 1 (AI):    6059
  - Tổng cộng:       12118
Tập validation:
  - Label 0 (Người): 2010
  - Label 1 (AI):    2010
  - Tổng cộng:       4020
Tập test:
  - Label 0 (Người): 5
  - Label 1 (AI):    54
  - Tổng cộng:       59


In [ ]:
class DataProcessor:
    def __init__(self, config):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config.model_name)
        self.stride = getattr(config.Tokenizer, 'stride', 64)
        self.max_length = getattr(config.Tokenizer, 'max_length', 256)
        self.truncation = False

    def tokenize_fn(self, examples):
        tokenized_inputs = self.tokenizer(
            examples["segmented_text"],
            add_special_tokens=False,
            truncation=False,
            padding=False
        )

        chunked_input_ids = []
        chunked_attention_mask = []
        chunked_labels = []
        chunked_texts = []
        chunked_doc_ids = []

        cls_token_id = self.tokenizer.cls_token_id
        sep_token_id = self.tokenizer.sep_token_id
        pad_token_id = self.tokenizer.pad_token_id

        chunk_size = self.max_length - 2
        step = max(1, chunk_size - self.stride)

        for i, input_ids in enumerate(tokenized_inputs["input_ids"]):
            doc_id = examples["doc_id"][i] if "doc_id" in examples else (examples["id"][i] if "id" in examples else i)
            label = examples["label"][i] if "label" in examples else None
            text = examples["segmented_text"][i] if "segmented_text" in examples else None

            if len(input_ids) <= chunk_size:
                chunks = [input_ids]
            else:
                chunks = [input_ids[j : j + chunk_size] for j in range(0, len(input_ids), step)]

            for chunk in chunks:
                final_ids = [cls_token_id] + chunk + [sep_token_id]
                final_mask = [1] * len(final_ids)

                pad_len = self.max_length - len(final_ids)
                if pad_len > 0:
                    final_ids.extend([pad_token_id] * pad_len)
                    final_mask.extend([0] * pad_len)

                chunked_input_ids.append(final_ids)
                chunked_attention_mask.append(final_mask)

                if label is not None:
                    chunked_labels.append(label)
                if text is not None:
                    chunked_texts.append(text)
                if doc_id is not None:
                    chunked_doc_ids.append(doc_id)

        res = {
            "input_ids": chunked_input_ids,
            "attention_mask": chunked_attention_mask,
        }

        if chunked_labels:
            res["label"] = chunked_labels
        if chunked_texts:
            res["segmented_text"] = chunked_texts
        if chunked_doc_ids:
            res["doc_id"] = chunked_doc_ids

        return res

    def process(self, dataset_dict):
        for split in dataset_dict.keys():
            original_columns = dataset_dict[split].column_names
            break

        processed_ds = dataset_dict.map(
            self.tokenize_fn,
            batched=True,
            remove_columns=original_columns
        )

        return processed_ds

    def finalize_datasets(self, tokenized_ds):
        target_columns = ["input_ids", "attention_mask", "label", "segmented_text", "doc_id"]

        for split in tokenized_ds.keys():
            existing_columns = [col for col in target_columns if col in tokenized_ds[split].column_names]

            tokenized_ds[split].set_format(
                type="torch",
                columns=existing_columns,
                output_all_columns=True
            )

        return tokenized_ds

In [ ]:
class PhoBertMetrics:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def compute_metrics(self, eval_pred):
        logits = eval_pred.predictions
        labels = eval_pred.label_ids

        if isinstance(logits, tuple):
            logits = logits[0]

        probs = softmax(logits, axis=-1)
        y_pred = np.argmax(probs, axis=-1)
        y_true = labels
        is_binary = probs.shape[-1] == 2
        prob_pos = probs[:, 1] if is_binary else probs
        precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
        recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
        f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
        accuracy = (y_pred == y_true).mean()
        mcc = matthews_corrcoef(y_true, y_pred)

        try:
            if is_binary:
                auc = roc_auc_score(y_true, prob_pos)
            else:
                auc = roc_auc_score(y_true, probs, multi_class='ovr')
        except ValueError:
            auc = 0.0

        if is_binary:
            brier = brier_score_loss(y_true, prob_pos)
            prob_true, prob_pred = calibration_curve(y_true, prob_pos, n_bins=10)
        else:
            brier = 0.0
            prob_true, prob_pred = np.array([]), np.array([])

        metrics_dict = {
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "mcc": mcc,
            "auc": auc,
            "brier_score": brier,
        }

        metrics_dict["calib_prob_true"] = prob_true.tolist()
        metrics_dict["calib_prob_pred"] = prob_pred.tolist()

        return metrics_dict

In [ ]:
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        if self.class_weights is not None:
            weight = torch.tensor(self.class_weights, device=model.device, dtype=torch.float)
            loss_fct = nn.CrossEntropyLoss(weight=weight)
        else:
            loss_fct = nn.CrossEntropyLoss()

        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [ ]:
class QuietEarlyStoppingCallback(EarlyStoppingCallback):
    def on_evaluate(self, args, state, control, metrics, **kwargs):
        metric_to_check = args.metric_for_best_model

        if metric_to_check and not metric_to_check.startswith("eval_"):
            metric_to_check = f"eval_{metric_to_check}"

        if metric_to_check not in metrics:
            return control

        return super().on_evaluate(args, state, control, metrics, **kwargs)

In [ ]:
class PhoBertTrainer:
    def __init__(self, config, model, train_ds, val_ds, tokenizer):
        self.config = config
        self.model = model
        self.train_ds = train_ds
        self.val_ds = val_ds
        self.tokenizer = tokenizer

    def apply_freeze(self, freeze_layers=0):
        if freeze_layers == -1:
            for param in self.model.base_model.parameters():
                param.requires_grad = False
        elif freeze_layers > 0:
            for param in self.model.base_model.embeddings.parameters():
                param.requires_grad = False

            for name, param in self.model.base_model.encoder.layer.named_parameters():
                layer_num = int(name.split('.')[0])
                if layer_num < freeze_layers:
                    param.requires_grad = False

        trainable = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.model.parameters())
        print(f"[Freeze] Trainable params: {trainable:,} / {total:,} ({trainable/total:.2%})")

    def train(self):
        training_args = TrainingArguments(
            output_dir=self.config.save_dir,
            num_train_epochs=self.config.Train.epochs,
            per_device_train_batch_size=self.config.Train.batch_size,
            per_device_eval_batch_size=self.config.Inference.batch_size,
            learning_rate=self.config.Train.lr,
            weight_decay=self.config.Train.weight_decay,
            eval_strategy="epoch",
            save_strategy="epoch",
            metric_for_best_model=self.config.Train.metric_for_best_model,
            load_best_model_at_end=True,
            greater_is_better=True,
            seed=self.config.seed,
            logging_steps=20,
            gradient_accumulation_steps=2,
            fp16=True,
            lr_scheduler_type=self.config.Train.lr_scheduler_type,
            warmup_ratio=self.config.Train.warmup_ratio,
            push_to_hub=self.config.Hub.push_to_hub,
            hub_model_id=self.config.Hub.repo_id,
            hub_token=self.config.Hub.token
        )

        metric_engine = PhoBertMetrics(tokenizer=self.tokenizer)

        trainer = WeightedTrainer(
            class_weights=self.config.Train.class_weights,
            model=self.model,
            args=training_args,
            train_dataset=self.train_ds,
            eval_dataset=self.val_ds,
            compute_metrics=metric_engine.compute_metrics,
            callbacks=[QuietEarlyStoppingCallback(early_stopping_patience=2)]
        )

        print("--- Bắt đầu huấn luyện ---")
        trainer.train()

        print(f"--- Lưu mô hình cục bộ tại {self.config.save_dir} ---")
        trainer.save_model(self.config.save_dir)
        self.tokenizer.save_pretrained(self.config.save_dir)

        if self.config.Hub.push_to_hub:
            print("--- Đang đẩy mô hình cuối cùng lên Hugging Face Hub ---")
            trainer.push_to_hub()
            print("--- Hoàn tất tải lên! ---")

        print("--- Huấn luyện kết thúc ---")
        return trainer

In [ ]:
class PhoBertInference:
    def __init__(self, config, model, tokenizer):
        self.config = config
        self.model = model
        self.tokenizer = tokenizer
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.model.eval()
        self.best_threshold = 0.5
        self.max_length = 256
        self.stride = getattr(config.Train, 'stride', 64)

    def predict_batch(self, tokenized_test_ds):
        test_loader = DataLoader(
            tokenized_test_ds,
            batch_size=self.config.Inference.batch_size
        )

        all_probs = []
        all_labels = []
        all_doc_ids = []

        with torch.no_grad():
            for batch in test_loader:
                inputs = {
                    "input_ids": batch["input_ids"].to(self.device),
                    "attention_mask": batch["attention_mask"].to(self.device)
                }

                outputs = self.model(**inputs)
                logits = outputs.logits if hasattr(outputs, 'logits') else outputs["logits"]

                probs = torch.softmax(logits, dim=-1)
                ai_probs = probs[:, 1].cpu().numpy()

                all_probs.extend(ai_probs)
                all_labels.extend(batch["label"].cpu().numpy())
                all_doc_ids.extend(batch["doc_id"])

        df = pd.DataFrame({
            "doc_id": all_doc_ids,
            "prob_ai": all_probs,
            "true_label": all_labels
        })

        doc_df = df.groupby('doc_id').agg({
            'prob_ai': 'mean',
            'true_label': 'first'
        }).reset_index()

        return doc_df["prob_ai"].values, doc_df["true_label"].values, doc_df["doc_id"].values

    def find_best_threshold(self, y_true, probs, optimize_for="mcc"):
        precision, recall, thresholds = precision_recall_curve(y_true, probs)

        if optimize_for == "f1":
            precision = precision[:-1]
            recall = recall[:-1]
            scores = (2 * precision * recall) / (precision + recall + 1e-8)

            best_idx = np.argmax(scores)
            self.best_threshold = thresholds[best_idx]
            print(f"\n[Threshold] Tìm thấy ngưỡng tối ưu (Best F1 = {scores[best_idx]:.4f}) tại: {self.best_threshold:.4f}")

        elif optimize_for == "mcc":
            mcc_scores = []

            for t in thresholds:
                y_pred = (probs >= t).astype(int)
                score = matthews_corrcoef(y_true, y_pred)
                mcc_scores.append(score)

            best_idx = np.argmax(mcc_scores)
            self.best_threshold = thresholds[best_idx]
            print(f"\n[Threshold] Tìm thấy ngưỡng tối ưu (Best MCC = {mcc_scores[best_idx]:.4f}) tại: {self.best_threshold:.4f}")

        return self.best_threshold

    def predict_text(self, text, threshold=None):
        if threshold is None:
            threshold = self.best_threshold

        tokenized = self.tokenizer(text, add_special_tokens=False, truncation=False, padding=False)
        input_ids = tokenized["input_ids"]

        chunk_size = self.max_length - 2
        step = max(1, chunk_size - self.stride)

        chunks = [input_ids] if len(input_ids) <= chunk_size else [input_ids[j : j + chunk_size] for j in range(0, len(input_ids), step)]

        cls_id = self.tokenizer.cls_token_id
        sep_id = self.tokenizer.sep_token_id
        pad_id = self.tokenizer.pad_token_id

        final_input_ids, final_masks = [], []

        for chunk in chunks:
            ids = [cls_id] + chunk + [sep_id]
            mask = [1] * len(ids)
            pad_len = self.max_length - len(ids)
            if pad_len > 0:
                ids.extend([pad_id] * pad_len)
                mask.extend([0] * pad_len)
            final_input_ids.append(ids)
            final_masks.append(mask)

        inputs = {
            "input_ids": torch.tensor(final_input_ids).to(self.device),
            "attention_mask": torch.tensor(final_masks).to(self.device)
        }

        with torch.no_grad():
            outputs = self.model(**inputs)
            logits = outputs.logits if hasattr(outputs, 'logits') else outputs["logits"]
            probs = torch.softmax(logits, dim=-1)
            ai_probs = probs[:, 1].cpu().numpy()

        sorted_probs = np.sort(ai_probs)[::-1]
        final_prob = float(np.mean(sorted_probs[:2])) if len(sorted_probs) > 1 else float(sorted_probs[0])

        return {
            "label": "AI tạo sinh" if label == 1 else "Người viết",
            "ai_probability": round(final_prob, 4),
            "threshold": round(threshold, 4),
            "num_chunks_processed": len(chunks)
        }

    def full_report(self, raw_dataset, probs, y_true, doc_ids):
        print("\n========== FULL DEBUG REPORT ==========")
        threshold = self.best_threshold
        y_pred = (probs > threshold).astype(int)
        print(f"Threshold đang dùng: {threshold:.4f}")

        print("\n--- 1. Classification Report ---")
        print(classification_report(y_true, y_pred, target_names=["Người viết", "AI tạo sinh"]))

        print("\n--- 2. Advanced Metrics ---")
        mcc = matthews_corrcoef(y_true, y_pred)
        auc = roc_auc_score(y_true, probs)
        print(f"MCC Score: {mcc:.4f} (Càng gần 1 càng tốt)")
        print(f"AUC-ROC Score: {auc:.4f}")

        print("\n--- 3. Confusion Matrix ---")
        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(6, 4))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=["Dự đoán Người", "Dự đoán AI"],
                    yticklabels=["Thực tế Người", "Thực tế AI"])
        plt.title("Ma trận nhầm lẫn")
        plt.show()

        print("\n--- 4. Phân tích lỗi (Sai lầm của mô hình) ---")
        wrong_idx = np.where(y_pred != y_true)[0]
        print(f"Tổng số ca dự đoán sai: {len(wrong_idx)} / {len(y_true)}")

        for idx in wrong_idx[:3]:
            real_doc_id = doc_ids[idx]
            row_idx = int(str(real_doc_id).split('_')[-1])
            original_text = raw_dataset[row_idx]["segmented_text"]

            print(f"\n[Doc ID: {real_doc_id}]")
            print(f"Text snippet: {original_text[:250]}...")
            print(f"True Label: {'AI' if y_true[idx]==1 else 'Người'} | Pred Label: {'AI' if y_pred[idx]==1 else 'Người'} | Prob: {probs[idx]:.4f}")
        print("-" * 50)

        print("\n--- 5. Biểu đồ đánh giá ngưỡng ---")
        precision, recall, thresholds = precision_recall_curve(y_true, probs)
        precision, recall = precision[:-1], recall[:-1]
        f1_scores = (2 * precision * recall) / (precision + recall + 1e-8)

        plt.figure(figsize=(12, 4))

        plt.subplot(1, 2, 1)
        plt.hist(probs[y_true == 0], bins=50, alpha=0.5, label="Người")
        plt.hist(probs[y_true == 1], bins=50, alpha=0.5, label="AI")
        plt.axvline(threshold, color='red', linestyle='--', label='Threshold')
        plt.legend()
        plt.title("Phân phối xác suất Document Level")

        plt.subplot(1, 2, 2)
        plt.plot(thresholds, precision, label="Precision")
        plt.plot(thresholds, recall, label="Recall")
        plt.plot(thresholds, f1_scores, label="F1-score")
        plt.axvline(threshold, color='red', linestyle='--', label=f'Best ({threshold:.2f})')
        plt.xlabel("Threshold")
        plt.title("Tối ưu hóa Threshold")
        plt.legend()
        plt.grid()

        plt.tight_layout()
        plt.show()
        print("\n========== END REPORT ==========")

In [ ]:
class AIDetectionPipeline:
    def __init__(self, config):
        self.config = config
        self.vncorenlp_dir = os.path.abspath('./vncorenlp_model')

        if not os.path.exists(self.vncorenlp_dir):
            os.makedirs(self.vncorenlp_dir)
            py_vncorenlp.download_model(save_dir=self.vncorenlp_dir)

        self.rdrsegmenter = py_vncorenlp.VnCoreNLP(
            annotators=["wseg"],
            save_dir=self.vncorenlp_dir
        )

    def segment_text(self, example):
        text_input = example.get('text', example.get('segmented_text', ''))
        sentences = self.rdrsegmenter.word_segment(text_input)
        example['segmented_text'] = " ".join(sentences)
        return example

    def run(self):
        print("--- [1] Đang xử lý dữ liệu ---")
        raw_dataset = load_dataset("kngann2201/vietnamese-ai-detection")

        print("Đang thực hiện Word Segmentation với VnCoreNLP...")
        raw_dataset = raw_dataset.map(self.segment_text)

        main_train_ds = raw_dataset["train"].map(lambda example, idx: {'doc_id': f"train_{idx}"}, with_indices=True)
        train_val_split = main_train_ds.train_test_split(test_size=0.1, seed=self.config.seed)

        ood_val_ds = raw_dataset["validation"].map(lambda example, idx: {'doc_id': f"ood_val_{idx}"}, with_indices=True)
        ood_test_ds = raw_dataset["test"].map(lambda example, idx: {'doc_id': f"ood_test_{idx}"}, with_indices=True)

        safe_dataset = DatasetDict({
            'train': train_val_split['train'],
            'validation': train_val_split['test'],
            'test_ood_1': ood_val_ds,
            'test_ood_2': ood_test_ds
        })

        processor = DataProcessor(self.config)
        tokenized_datasets = processor.process(safe_dataset)
        tokenized_datasets = processor.finalize_datasets(tokenized_datasets)

        eval_datasets_dict = {
            "id_val": tokenized_datasets["validation"],
            "ood1_val": tokenized_datasets["test_ood_1"]
        }

        print(f"--- [2] Khởi tạo mô hình {self.config.model_name} ---")
        model = AutoModelForSequenceClassification.from_pretrained(
            self.config.model_name,
            num_labels=2
        )

        print("--- [3] Cấu hình Trainer & Bắt đầu huấn luyện ---")
        trainer_engine = PhoBertTrainer(
            config=self.config,
            model=model,
            train_ds=tokenized_datasets["train"],
            val_ds=eval_datasets_dict,
            tokenizer=processor.tokenizer
        )

        trainer_engine.apply_freeze(self.config.Train.freeze_layers)
        trainer = trainer_engine.train()

        print("--- [Hoàn thành Pipeline AI Detection!] ---")
        return trainer


In [ ]:
config = MyConfig()
pipeline = AIDetectionPipeline(config)
pipeline.run()

--- [1] Đang xử lý dữ liệu ---
Đang thực hiện Word Segmentation với VnCoreNLP...


Parameter 'function'=<bound method AIDetectionPipeline.segment_text of <__main__.AIDetectionPipeline object at 0x7b9baca71fa0>> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/12118 [00:00<?, ? examples/s]

Map:   0%|          | 0/4020 [00:00<?, ? examples/s]

Map:   0%|          | 0/59 [00:00<?, ? examples/s]

...

New Data Upload               : |          |  0.00B /  0.00B            

  ...etector/training_args.bin: 100%|##########| 5.26kB / 5.26kB            

  ...etector/model.safetensors:   3%|2         | 40.0MB / 1.48GB            

No files have been modified since last commit. Skipping to prevent empty commit.


--- Hoàn tất tải lên! ---
--- Huấn luyện kết thúc ---
--- [Hoàn thành Pipeline AI Detection!] ---
